# MJO 8: Calculate MJO Plots

Via `mjo.ncl`

- Read daily output files from CAM2 and process the u200 data
- Each daily file has 30 days of data
- U200 (time, lat, lon)

In [1]:
# # getenv == os.environ
import os

CASENAME = "QBOi.EXP1.AMIP.001"
startdate = '19790101'
enddate = '19811231'

# /glade/u/home/bundy/mdtf/MDTF_3_main/MDTF-diagnostics.blocking_notebook/diagnostics/MJO_suite/MJO_driver.py
#DATADIR = "/glade/u/home/bundy/diag/mdtf/inputdata/model/QBOi.EXP1.AMIP.001/"
WORK_DIR = os.getcwd()
DATADIR = "/data/"

U200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U200.day.nc"
V200_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V200.day.nc"
U850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.U850.day.nc"
V850_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.V850.day.nc"
RLUT_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.FLUT.day.nc"
PR_FILE = WORK_DIR + DATADIR + "QBOi.EXP1.AMIP.001.PRECT.day.nc"

lev_coord = "lev"
lat_coord = "lat"
lon_coord = "lon"
time_coord = "time"
pr_var = "PRECT"
rlut_var = "FLUT"

u200_var = "U200"
v200_var = "V200"

wk_dir = WORK_DIR + "/model/"

# setup data directory, if does not already exist
#if not os.path.exists(DATADIR): os.makedirs(DATADIR)

In [2]:
from datetime import datetime, timedelta

firstyr = datetime.strptime(startdate, "%Y%m%d")
lastyr =  datetime.strptime(enddate, "%Y%m%d")

print(firstyr)
print(lastyr)

ndays = abs((firstyr - lastyr).days) # L26 (mjo.ncl)
print(f"Days between start and end: {ndays}")

1979-01-01 00:00:00
1981-12-31 00:00:00
Days between start and end: 1095


In [3]:
import xarray as xr
file_u200 = xr.open_dataset(WORK_DIR + DATADIR + CASENAME + ".U200.day.nc") # L28 (mjo.ncl)
file_u200

<xarray.Dataset> Size: 565MB
Dimensions:    (time: 2555, nbnd: 2, lat: 192, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    U200       (time, lat, lon) float32 565MB ...

In [4]:
date = file_u200["date"]
maxdays = len(date)
print(f"Max Days = {maxdays}")

if ndays > maxdays: # L32 (mjo.ncl)
    ndays = maxdays

Max Days = 2555


In [5]:
# L35 (mjo.ncl)
u200 = file_u200.isel(time=slice(0, maxdays))
u200 = u200.sel(lat=slice(-10, 10)) 
u200

<xarray.Dataset> Size: 65MB
Dimensions:    (time: 2555, nbnd: 2, lat: 22, lon: 288)
Coordinates:
  * time       (time) object 20kB 1975-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat        (lat) float64 176B -9.895 -8.953 -8.01 ... 8.01 8.953 9.895
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.3 357.5 358.8
Dimensions without coordinates: nbnd
Data variables:
    date       (time) int32 10kB ...
    time_bnds  (time, nbnd) object 41kB ...
    U200       (time, lat, lon) float32 65MB ...

In [6]:
yrfrac_time = file_u200["time"] # L28 (mjo.ncl)
yrfrac = yrfrac_time.isel(time=slice(0, ndays-1))

year_date = yrfrac.dt.strftime("%Y")
year_date

<xarray.DataArray 'strftime' (time: 1094)> Size: 9kB
array(['1975', '1975', '1975', ..., '1977', '1977', '1977'],
      shape=(1094,), dtype=object)
Coordinates:
  * time     (time) object 9kB 1975-01-01 00:00:00 ... 1977-12-30 00:00:00

In [7]:
fraction_of_year = (yrfrac.dt.dayofyear/365).astype(str)
fraction_of_year = fraction_of_year.str.replace(r"^0", "") # remove preceding 0. from 0.1
fraction_of_year

<xarray.DataArray 'dayofyear' (time: 1094)> Size: 88kB
array(['.0027397260273972603', '.005479452054794521',
       '.00821917808219178', ..., '.9917808219178083',
       '.9945205479452055', '.9972602739726028'],
      shape=(1094,), dtype='<U20')
Coordinates:
  * time     (time) object 9kB 1975-01-01 00:00:00 ... 1977-12-30 00:00:00

In [8]:
# YYYY.fraction_of_year
yrfrac = year_date.str.cat(fraction_of_year)
yrfrac = yrfrac.rename({list(yrfrac.coords.keys())[0]: "date as fraction of year"}) #L40 (mjo.ncl)
yrfrac

<xarray.DataArray (date as fraction of year: 1094)> Size: 9kB
array(['1975.0027397260273972603', '1975.005479452054794521',
       '1975.00821917808219178', ..., '1977.9917808219178083',
       '1977.9945205479452055', '1977.9972602739726028'],
      shape=(1094,), dtype=object)
Coordinates:
  * date as fraction of year  (date as fraction of year) object 9kB 1975-01-0...

In [20]:
# 1_MJO_daily_netcdf.ipynb

def read_dim(f, var_name, opt=None, upper_bound=None, lower_bound=None):
    print("running read_dim")
    # filter lat bounds from 40 to -40
    routine_name = "read_dim"
    verbose = False
    if opt:
        if lower_bound is not None:
            #print(f"found lower_bound {lower_bound}")
            pass
        if upper_bound is not None:
            #print(f"found upper_bound {upper_bound}")
            pass
    
    if lower_bound is not None and upper_bound is not None:
        # filter based on upper and lower bound
        var = f[var_name].where((f[var_name] > lower_bound) & (f[var_name] < upper_bound))
    else:
        var = f[var_name]

    return var    

def get_gw(f, latS, latN):
    # gaussian weight for the latitude dimension
    if False:
    #if "gw" in file_netcdf.coords:
        # remove any existing "gw" dimension
        #f.drops_vars("gw")
        #new_gw = f[gw[lat_coord|latS:latN]]
        print("TODO")
    else:
        print("running get_gw")
        lat = read_dim(f, lat_coord, opt=False, upper_bound=latN, lower_bound=latS) # cut full range based on upper and lower bounds
        nlat = lat.count().item()
        slat = xr.DataArray(coords=(range(nlat+1), ))
        #gw = xr.DataArray(coords=(range(nlat+1), ), dims="gw")
        new_gw = {}

        if lat[0].values < lat[1].values:
            slat[0] = -90.0
            slat[nlat] = 90.0
        else:
            slat[0] = 90.0
            slat[nlat] = -90.0

        for i in range(nlat-1):
            slat[i] = (lat[i-1].values + lat[i].values)/2

        for i in range(nlat-1):
            new_gw[i] = abs(np.sin(slat[i+1] / (180*np.pi)) - np.sin(slat[i] / (180 * np.pi)))
        
    return new_gw

In [27]:
weights_lat = get_gw(u200, -10, 10) #L29 (mjo.ncl)
weights_lon = np.ones(len(u200["lon"]))
date = u200 # L30 (mjo.ncl)

running get_gw
running read_dim


In [33]:
# compute weighted area average of U200 over tropics
import numpy as np

def wgt_areaave(data, weights_lat=None, weights_lon=None):
    # weight arrays
    if weights_lat is None:
        weights_lat = np.ones(len(data["lat"]))
    if weights_lon is None:
        weights_lon = np.ones(len(data["lon"]))

    # setup weights as xarray
    print(weights_lat)
    lat_weights_xr = xr.DataArray(weights_lat, coords={"lat": data["lat"]}, dims="lat")
    lon_weights_xr = xr.DataArray(weights_lon, coords={"lon": data["lon"]}, dims="lon")

    # weights in 2D
    weights = lat_weights_xr * lon_weights_xr

    # sum weights
    weighted_data = data * weights

    # weighted result
    result = weighted_data.sum(dim=["lat", "lon"], skipna=True) / weights.where(~np.isnan(data)).sum()

    return result

In [34]:
data = wgt_areaave(u200, weights_lat, weights_lon) # L45 (mjo.ncl)
data

{0: <xarray.DataArray ()> Size: 8B
array(0.01666467), 1: <xarray.DataArray ()> Size: 8B
array(0.00166633), 2: <xarray.DataArray ()> Size: 8B
array(0.00166638), 3: <xarray.DataArray ()> Size: 8B
array(0.00166641), 4: <xarray.DataArray ()> Size: 8B
array(0.00166645), 5: <xarray.DataArray ()> Size: 8B
array(0.00166647), 6: <xarray.DataArray ()> Size: 8B
array(0.0016665), 7: <xarray.DataArray ()> Size: 8B
array(0.00166652), 8: <xarray.DataArray ()> Size: 8B
array(0.00166653), 9: <xarray.DataArray ()> Size: 8B
array(0.00166654), 10: <xarray.DataArray ()> Size: 8B
array(0.00166654), 11: <xarray.DataArray ()> Size: 8B
array(0.00166654), 12: <xarray.DataArray ()> Size: 8B
array(0.00166654), 13: <xarray.DataArray ()> Size: 8B
array(0.00166653), 14: <xarray.DataArray ()> Size: 8B
array(0.00166652), 15: <xarray.DataArray ()> Size: 8B
array(0.0016665), 16: <xarray.DataArray ()> Size: 8B
array(0.00166647), 17: <xarray.DataArray ()> Size: 8B
array(0.00166645), 18: <xarray.DataArray ()> Size: 8B
arra

ValueError: different number of dimensions on data and dims: 0 vs 1